# Experiment 4 - Part 1: Building the Knowledge Base for RAG

**Course:** HAAI302COM - Artificial Intelligence Laboratory

A language model only knows what was in its training data. Ask it about your own PDFs and it will either refuse or, worse, invent an answer. **Retrieval-Augmented Generation (RAG)** fixes this by looking the answer up first and then asking the model to write it using only what was found.

```
PDF files -> Load -> Split into chunks -> Embed -> Store in a vector DB     (this notebook)
Question  -> Embed -> Retrieve top-k chunks -> Prompt LLM -> Answer         (notebook 02)
Question + answer + chunks -> RAGAs metrics                                 (notebook 03)
```

In this notebook we build the left half of the pipeline, one library call at a time, and look inside every object we create.

**Our documents:** two real research papers, the original RAG paper (Lewis et al., 2020) and the LoRA paper (Hu et al., 2021) that you met in Experiment 2.

## 0. Setup

If you run this on Colab, uncomment the install line first. Locally everything is already installed.

We use **LangChain** for loading, splitting and the vector store wrapper, **sentence-transformers** (a small model that runs on CPU, no API key) for embeddings, and **ChromaDB** as the vector database.

In [1]:
# !pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-text-splitters chromadb pypdf sentence-transformers ragas
import os, re, json, shutil, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 1. Loading PDFs

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page. A `Document` has just two fields: `page_content` (the text) and `metadata` (a dictionary that travels with the text everywhere, which is how we will later show citations).

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pages = []
for name in ["rag_lewis_2020.pdf", "lora_hu_2021.pdf"]:
    pages += PyPDFLoader(f"data/{name}").load()   # one Document per page

print(len(pages), "pages loaded")
print(type(pages[0]).__name__, list(pages[0].model_fields))

45 pages loaded
Document ['id', 'metadata', 'page_content', 'type']


In [3]:
# metadata added by the loader
pages[0].metadata

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2021-04-13T00:48:38+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2021-04-13T00:48:38+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'data/rag_lewis_2020.pdf',
 'total_pages': 19,
 'page': 0,
 'page_label': '1'}

In [4]:
# one row per page: where it came from and how much text it holds
page_df = pd.DataFrame({
    "source": [os.path.basename(d.metadata["source"]) for d in pages],
    "page": [d.metadata["page"] for d in pages],
    "chars": [len(d.page_content) for d in pages],
    "words": [len(d.page_content.split()) for d in pages],
})
page_df.groupby("source")[["chars", "words"]].agg(["count", "sum", "mean"]).round(0)

chars                words              
                   count    sum    mean count    sum   mean
source                                                     
lora_hu_2021.pdf      26  82471  3172.0    26  13347  513.0
rag_lewis_2020.pdf    19  69058  3635.0    19   9885  520.0

**What to notice:** the RAG paper has 19 pages and the LoRA paper 26, with roughly 4,000 to 5,000 words per page of dense text. In total we have around 45 pages, which is far more than fits comfortably in one prompt for a cheap model, and certainly not something we want to pay to resend on every question.

PDF text is rarely clean. Let us look at a raw sample.

In [5]:
raw = pages[1].page_content
i = raw.index("\ufb01")                      # position of the first ligature on this page
print(raw[i - 150 : i + 250])

d	into	3
sections:	"Inferno",
"Purgatorio"	&
"Paradiso"									(y)
End-to-End Backprop through q and pθ
Barack	Obama	was
born	in	Hawaii.(x)
Fact Veriﬁcation: Fact Query
supports	(y)
Question Generation
Fact Veriﬁcation:
Label Generation
Document
Index
Define	"middle	ear"(x)
Question Answering:
Question Query
The	middle	ear	includes
the	tympanic	cavity	and
the	three	ossicles.		(y)
Question Answeri


**What to notice:** this page holds a figure, and its labels come out as fragments separated by tabs, a typical PDF problem. Also look at `Veriﬁcation`: it contains the single ligature character `ﬁ` instead of the two letters `fi`. If we leave it, a query typed as "verification" or "fine-tuning" will not match the text and an embedding model may treat the words differently. In body text, lines also break mid-sentence because the paper is laid out in two columns.

We apply a small cleaning function: replace ligatures and collapse whitespace. We do **not** try to be clever beyond that, because over-cleaning PDFs is a source of bugs.

In [6]:
LIGATURES = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl"}

def clean(text):
    for bad, good in LIGATURES.items():
        text = text.replace(bad, good)          # ligature glyph -> plain letters
    return re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces

before = sum(d.page_content.count("ﬁ") for d in pages)
for d in pages:
    d.page_content = clean(d.page_content)
after = sum(d.page_content.count("ﬁ") for d in pages)
print("ligature 'fi' characters before:", before, "| after:", after)

ligature 'fi' characters before: 317 | after: 0


## 2. Splitting into chunks

We cannot embed a whole page as one vector: the vector would be an average of many topics and a search result of 4,000 words is too big to read or to send to an LLM. So we cut pages into **chunks**.

`RecursiveCharacterTextSplitter` is the standard choice. It tries to split on the biggest natural boundary first (paragraph break, then line break, then space, then single character) and only falls back to a smaller boundary when a piece is still too large.

| Argument | What it controls |
|---|---|
| `chunk_size` | Maximum length of a chunk (in characters here, because `length_function=len`). |
| `chunk_overlap` | How many characters are repeated between neighbouring chunks, so a sentence cut at a boundary survives intact in one of them. |
| `separators` | The ordered list of boundaries to try. Default: `["\n\n", "\n", " ", ""]`. |
| `add_start_index` | Store the character offset of each chunk in its metadata. |

First a tiny example so you can see the mechanics before we use it on the papers.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

demo = ("RAG retrieves passages before generating an answer. The retriever is trained together with the generator.\n\n"
        "LoRA freezes the pre-trained weights. It trains small low-rank matrices instead. The update can be merged after training.")
demo_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=30)
for i, piece in enumerate(demo_splitter.split_text(demo)):
    print(i, len(piece), repr(piece))

0 94 'RAG retrieves passages before generating an answer. The retriever is trained together with the'
1 39 'is trained together with the generator.'
2 98 'LoRA freezes the pre-trained weights. It trains small low-rank matrices instead. The update can be'
3 49 'instead. The update can be merged after training.'


**What to notice:** the first cut falls on the blank line, so no chunk mixes RAG text with LoRA text. Each paragraph is longer than 100 characters, so it is cut again at a space. Chunk 1 starts with `is trained together with the` and chunk 3 starts with `instead. The update`, words that were also at the end of the chunk before them. That repeated stretch is the overlap, and its job is to keep a sentence that straddles a boundary readable in at least one chunk.

Now split the real pages. `split_documents` keeps each page's metadata and copies it onto every chunk.

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, add_start_index=True)
chunks = splitter.split_documents(pages)

print(len(pages), "pages ->", len(chunks), "chunks")
chunks[10].metadata

45 pages -> 237 chunks


{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2021-04-13T00:48:38+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2021-04-13T00:48:38+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'data/rag_lewis_2020.pdf',
 'total_pages': 19,
 'page': 1,
 'page_label': '2',
 'start_index': 3391}

In [9]:
chunk_df = pd.DataFrame({
    "source": [os.path.basename(c.metadata["source"]) for c in chunks],
    "page": [c.metadata["page"] for c in chunks],
    "start": [c.metadata["start_index"] for c in chunks],
    "chars": [len(c.page_content) for c in chunks],
    "text": [c.page_content[:70].replace("\n", " ") for c in chunks],
})
print(chunk_df["chars"].describe().round(0).to_string())
chunk_df.head(6)

count    237.0
mean     703.0
std      143.0
min      136.0
25%      717.0
50%      748.0
75%      779.0
max      799.0


,source,page,start,chars,text
0,rag_lewis_2020.pdf,0,0,768,Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patri
1,rag_lewis_2020.pdf,0,683,794,"lags behind task-specific architectures. Additionally, providing prove"
2,rag_lewis_2020.pdf,0,1399,790,"pare two RAG formulations, one which conditions on the same retrieved"
3,rag_lewis_2020.pdf,0,2093,771,edge from data [47]. They can do so without any access to an external
4,rag_lewis_2020.pdf,0,2767,136,"combine masked language models [8] with a differentiable retriever, ha"
5,rag_lewis_2020.pdf,1,0,723,The Divine Comedy (x) q Query Encoder q(x) MIPS pθ Generator pθ (Param


**What to notice:** chunk lengths cluster just under 800 because the splitter packs as many whole pieces as fit. Some chunks are much shorter (the last piece of a page, or a figure caption), and those short chunks are poor search targets.

Let us verify the overlap with our own eyes: the tail of one chunk should reappear at the head of the next.

In [10]:
a, b = chunks[10].page_content, chunks[11].page_content
print("END of chunk 10  :", repr(a[-110:]))
print("START of chunk 11:", repr(b[:110]))

END of chunk 10  : '1, our models\nleverage two components: (i) a retriever pη(z|x) with parametersη that returns (top-K truncated)'
START of chunk 11: 'leverage two components: (i) a retriever pη(z|x) with parametersη that returns (top-K truncated)\ndistributions'


### Experiment: how does `chunk_size` change the chunk count?

Changing one argument at a time is the quickest way to build intuition. We keep the overlap at 12 percent of the chunk size and vary only `chunk_size`.

In [11]:
rows = []
for size in [200, 400, 800, 1600, 3200]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.12))
    cs = sp.split_documents(pages)
    lens = [len(c.page_content) for c in cs]
    rows.append({"chunk_size": size, "n_chunks": len(cs), "mean_chars": round(np.mean(lens)), "min_chars": min(lens)})
pd.DataFrame(rows)

,chunk_size,n_chunks,mean_chars,min_chars
0,200,912,167,38
1,400,446,346,30
2,800,230,703,102
3,1600,124,1314,192
4,3200,72,2234,500


**What to notice:** halving the chunk size roughly doubles the number of chunks. Small chunks mean precise matches but each one carries little context. Large chunks carry context but dilute the topic. We will measure which is better for retrieval at the end of this notebook.

A second, less obvious limit: the embedding model itself can only read so many tokens.

## 3. Embeddings

An embedding model turns a text into a fixed-length vector so that texts with similar meaning land close together. We use `all-MiniLM-L6-v2` from sentence-transformers: small (22M parameters), fast on CPU, and free.

`HuggingFaceEmbeddings` is LangChain's wrapper. It exposes `embed_query` (one string) and `embed_documents` (a list of strings), which is the interface every vector store expects.

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
st_model = embedder._client                      # the underlying SentenceTransformer

print("max_seq_length (tokens):", st_model.max_seq_length)
print("vector size:", st_model.get_sentence_embedding_dimension())

max_seq_length (tokens): 256
vector size: 384


**What to notice:** `max_seq_length` is 256 tokens. Anything beyond that is silently **truncated** before embedding. So a 3,000-character chunk is embedded using only its first 200 words or so, and the rest is invisible to search. Let us check how our chunk sizes compare to that limit.

In [13]:
tok = st_model.tokenizer
sample = [c.page_content for c in chunks[:200]]
n_tok = [len(tok(t, verbose=False)["input_ids"]) for t in sample]   # verbose=False: no long-input warning
print("chunk_size=800: tokens per chunk -> mean", round(np.mean(n_tok)), "| max", max(n_tok))
print("share of chunks above 256 tokens:", round(np.mean(np.array(n_tok) > 256), 2))

chunk_size=800: tokens per chunk -> mean 189 | max 467
share of chunks above 256 tokens: 0.06


**What to notice:** at 800 characters most chunks are close to or below the 256-token limit, and a few longer ones (formulas, tables with many short tokens) get truncated. As a rule of thumb, 4 characters is about 1 token for English, so 800 characters is about 200 tokens. Chunk sizes much bigger than 1,000 characters would waste text with this particular embedder.

Now let us embed a query and see what comes out.

In [14]:
v = embedder.embed_query("What is low-rank adaptation?")
print(type(v).__name__, len(v))
print("first 5 values:", np.round(v[:5], 3))
print("vector length (L2 norm):", round(float(np.linalg.norm(v)), 4))

list 384
first 5 values: [-0.026 -0.051  0.029  0.047  0.063]
vector length (L2 norm): 1.0


**What to notice:** a plain list of 384 floats with a length of 1.0. The vector is **normalised**, so the dot product of two vectors equals their cosine similarity. That matters when we choose the distance metric in Chroma.

Let us check that similar meaning really gives similar vectors, using six hand-written sentences, two per topic (LoRA, RAG, cooking).

In [15]:
sents = [
    "LoRA freezes the pre-trained weights and trains small low-rank matrices.",
    "Low-rank adapters let us fine-tune a large model with few trainable parameters.",
    "RAG retrieves documents and feeds them to the generator.",
    "A retrieval-augmented model looks up passages before answering.",
    "Simmer the onions in butter until golden brown.",
    "Add salt to the pasta water and boil for ten minutes.",
]
E = np.array(embedder.embed_documents(sents))           # shape (6, 384)
sim = pd.DataFrame(E @ E.T, index=range(6), columns=range(6)).round(2)   # dot product = cosine
sim

,0,1,2,3,4,5
0,1.00,0.44,0.12,0.12,0.04,0.08
1,0.44,1.00,0.16,0.20,0.05,0.03
2,0.12,0.16,1.00,0.27,0.08,0.07
3,0.12,0.20,0.27,1.00,0.03,0.00
4,0.04,0.05,0.08,0.03,1.00,0.34
5,0.08,0.03,0.07,0.00,0.34,1.00


**What to notice:** read the table in blocks. The two LoRA sentences (0 and 1) score 0.44 with each other, the two RAG sentences (2 and 3) 0.27, and the two cooking sentences (4 and 5) 0.34. For every sentence, its partner beats every sentence from another topic. The scores are modest because the sentences share few words; what matters is the ordering. Also see the degrees of relatedness: a LoRA sentence vs a RAG sentence (0.12 to 0.20) scores higher than a LoRA sentence vs a cooking sentence (0.03 to 0.08), even though neither pair is a true match.

## 4. Storing vectors in ChromaDB

A vector database stores the vectors next to the original text and metadata, and answers "give me the k closest vectors to this query vector" quickly using an index (Chroma uses HNSW, an approximate nearest-neighbour graph).

`Chroma.from_documents` does three things in one call: embeds every chunk with our embedder, stores the vectors, text and metadata, and builds the index. Key arguments:

| Argument | Meaning |
|---|---|
| `collection_name` | A named table inside the database. |
| `persist_directory` | Folder on disk. Without it the database lives only in memory. |
| `collection_metadata={"hnsw:space": ...}` | Distance used by the index: `l2` (default), `cosine` or `ip` (inner product). |

In [16]:
from langchain_chroma import Chroma

DB_DIR = "chroma_db"
shutil.rmtree(DB_DIR, ignore_errors=True)              # start clean so re-runs are identical

store = Chroma.from_documents(
    chunks, embedder,
    collection_name="rag_papers",
    persist_directory=DB_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)
print("stored chunks:", store._collection.count())

stored chunks: 237


`store` is only a thin LangChain wrapper. The real Chroma collection is at `store._collection`, and we can read the raw records from it. This is how you debug a RAG system: look at exactly what was stored.

In [17]:
rec = store._collection.get(limit=3, include=["documents", "metadatas", "embeddings"])
print("keys:", list(rec))
print("ids:", rec["ids"])
print("embedding shape:", np.array(rec["embeddings"]).shape)

keys: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
ids: ['e0a02e87-2b01-4b48-89d6-15f0ae163923', 'a30dd827-196e-455c-89cb-69257e86456d', '58c4ca02-a908-4f79-afa3-90db812df06c']
embedding shape: (3, 384)


In [18]:
pd.DataFrame({
    "id": [i[:8] for i in rec["ids"]],
    "source": [m["source"].split("/")[-1] for m in rec["metadatas"]],
    "page": [m["page"] for m in rec["metadatas"]],
    "text": [d[:80].replace("\n", " ") for d in rec["documents"]],
})

,id,source,page,text
0,e0a02e87,rag_lewis_2020.pdf,0,Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡
1,a30dd827,rag_lewis_2020.pdf,0,"lags behind task-specific architectures. Additionally, providing provenance for"
2,58c4ca02,rag_lewis_2020.pdf,0,"pare two RAG formulations, one which conditions on the same retrieved passages a"


**What to notice:** every record has an auto-generated id, the original text, the metadata we carried through (source file and page) and a 384-dim vector. Because the metadata is stored, any answer we produce later can point back to "file X, page Y".

## 5. Searching

`similarity_search_with_score` embeds the query with the same embedder and returns the closest chunks together with their **distance**. For the cosine space, Chroma's distance is `1 - cosine_similarity`, so **smaller is better**.

In [19]:
query = "How does LoRA avoid adding inference latency?"
hits = store.similarity_search_with_score(query, k=4)

pd.DataFrame({
    "rank": range(1, 5),
    "distance": [round(s, 3) for _, s in hits],
    "cos_sim": [round(1 - s, 3) for _, s in hits],
    "source": [d.metadata["source"].split("/")[-1] for d, _ in hits],
    "page": [d.metadata["page"] for d, _ in hits],
    "text": [d.page_content[:70].replace("\n", " ") for d, _ in hits],
})

,rank,distance,cos_sim,source,page,text
0,1,0.353,0.647,lora_hu_2021.pdf,3,a subset of the pre-trained parameters. LoRA takes a step further and
1,2,0.401,0.599,lora_hu_2021.pdf,4,during training on GPT-3 175B compared to full fine-tuning 5 as we do
2,3,0.409,0.591,lora_hu_2021.pdf,11,networks with dense layers. There are many directions for future works
3,4,0.409,0.591,lora_hu_2021.pdf,20,"First of all, LoRA+PE significantly outperforms both LoRA and prefix-e"


**What to notice:** distances increase down the table (best match first) and all four hits come from the LoRA paper without us telling the database to look there. Read the top text yourself and confirm it really answers the question; a similarity score is a hint, not a guarantee.

### Experiment: the distance metric

We rebuild the same collection with `l2` (Euclidean) and `ip` (inner product) and compare the numbers and the ranking for the same query.

In [20]:
def top_ids(space, k=4):
    s = Chroma.from_documents(chunks, embedder, collection_name=f"space_{space}",
                              collection_metadata={"hnsw:space": space})   # in-memory
    return [(d.metadata["page"], round(sc, 3)) for d, sc in s.similarity_search_with_score(query, k=k)]

for space in ["cosine", "l2", "ip"]:
    print(f"{space:7s}", top_ids(space))

cosine  [(3, 0.353), (4, 0.401), (11, 0.409), (20, 0.409)]


l2      [(3, 0.707), (4, 0.803), (11, 0.817), (20, 0.818)]


ip      [(3, 0.353), (4, 0.401), (11, 0.409), (20, 0.409)]


**What to notice:** the numbers differ (cosine is `1 - sim`, ip is `1 - dot`, l2 is the squared Euclidean distance, here exactly twice the cosine distance: 0.707 vs 0.353), but the **same pages come back in the same order**. That is because our vectors are normalised: for unit vectors, squared L2 distance is `2 - 2 x cosine`, so all three metrics are monotonic in each other. With embedding models that do **not** normalise, the choice starts to matter.

## 6. Metadata filters

Because every chunk carries metadata, we can restrict a search to a subset *before* ranking. Chroma filters use operators such as `$eq`, `$gte`, `$and`. This is how you build "search only this document" or "only after page 5" features.

In [21]:
q2 = "What rank r is enough?"
for flt in [None, {"source": {"$eq": "data/rag_lewis_2020.pdf"}}]:
    res = store.similarity_search_with_score(q2, k=3, filter=flt)
    print("filter:", flt)
    for d, s in res:
        print("   ", d.metadata["source"].split("/")[-1], "p", d.metadata["page"], round(s, 3))

filter: None
    lora_hu_2021.pdf p 9 0.535
    lora_hu_2021.pdf p 9 0.559
    lora_hu_2021.pdf p 25 0.562
filter: {'source': {'$eq': 'data/rag_lewis_2020.pdf'}}
    rag_lewis_2020.pdf p 5 0.683
    rag_lewis_2020.pdf p 5 0.711
    rag_lewis_2020.pdf p 7 0.753


**What to notice:** without a filter, the best chunks for a LoRA question come from the LoRA paper. With the filter set to the RAG paper the database is forced to return the *least bad* RAG chunks, with clearly larger distances. A filter never creates relevance; it only restricts the candidate pool, so always inspect the distances.

## 7. Maximal Marginal Relevance (MMR)

Plain top-k often returns several near-duplicate chunks (the same idea repeated in the abstract, introduction and conclusion). **MMR** fetches a larger pool (`fetch_k`) and then picks `k` results that are relevant to the query *and* different from each other. `lambda_mult` slides between diversity (0) and pure relevance (1).

In [22]:
def show(docs):
    return [(d.metadata["source"].split("/")[-1][:4], d.metadata["page"]) for d in docs]

q3 = "How is the retriever trained and used?"
print("similarity   :", show(store.similarity_search(q3, k=5)))
for lam in [1.0, 0.8, 0.5]:
    docs = store.max_marginal_relevance_search(q3, k=5, fetch_k=20, lambda_mult=lam)
    print(f"mmr lambda={lam}:", show(docs))

similarity   : [('rag_', 5), ('rag_', 6), ('rag_', 2), ('rag_', 18), ('rag_', 4)]
mmr lambda=1.0: [('rag_', 5), ('rag_', 6), ('rag_', 2), ('rag_', 18), ('rag_', 4)]
mmr lambda=0.8: [('rag_', 5), ('rag_', 6), ('rag_', 2), ('lora', 11), ('lora', 8)]


mmr lambda=0.5: [('rag_', 5), ('lora', 11), ('rag_', 7), ('lora', 8), ('lora', 17)]


**What to notice:** with `lambda_mult=1.0` MMR is identical to plain similarity (all five from the RAG paper, pages 5, 6, 2, 18, 4). At 0.8 the last two slots are replaced by LoRA chunks, and at 0.5 only two of five are from the RAG paper. MMR is doing its job (more variety), but here the variety is **off-topic**: our corpus holds only a handful of chunks about the retriever, and once they start to resemble each other MMR prefers an unrelated chunk over a second useful one. Use MMR when you have many near-duplicate hits; keep `lambda_mult` high (0.7 to 1.0) so relevance still dominates, and always look at what comes back.

## 8. Does chunk size actually matter for retrieval? A quick measurement

Before involving any LLM, we can measure retrieval on its own. We have 12 questions about the two papers (`qa_set.json`) and, for each, a **keyword** that must appear in a chunk for that chunk to count as containing the answer. The metric is **hit@k**: the fraction of questions where at least one of the top-k chunks contains the keyword.

This is a crude proxy (a chunk can contain the keyword without really answering), but it needs no LLM and is a good first sanity check.

In [23]:
qa = json.load(open("qa_set.json"))["answerable"]
pd.DataFrame(qa)[["source", "question", "keyword"]].assign(question=lambda d: d.question.str[:75])

,source,question,keyword
0,rag_lewis_2020.pdf,"What are the two RAG formulations compared in the RAG paper, and how do the",RAG-Token
1,rag_lewis_2020.pdf,"Which retriever does the RAG model use, and what architecture is it?",bi-encoder
2,rag_lewis_2020.pdf,Which generator model does RAG use and how many parameters does it have?,BART-large
3,rag_lewis_2020.pdf,How was the Wikipedia knowledge source prepared for the RAG index?,21M
4,rag_lewis_2020.pdf,What does the index hot-swapping experiment show in the RAG paper?,hot-swapping
5,rag_lewis_2020.pdf,What search problem is solved to find the top-K documents in RAG?,Maximum Inner Product Search
6,lora_hu_2021.pdf,By how much does LoRA reduce trainable parameters and GPU memory for GPT-3,"10,000 times"
7,lora_hu_2021.pdf,Which weight matrices does LoRA adapt in most of its experiments?,Wq and Wv
8,lora_hu_2021.pdf,"How are the LoRA matrices A and B initialised, and why does it matter?",random Gaussian initialization
9,lora_hu_2021.pdf,Why does LoRA add no inference latency?,explicitly compute and store


In [24]:
def norm(t):
    return re.sub(r"\s+", " ", t).lower()               # ignore line breaks and case

def hit_rates(size, ks=(1, 4), overlap_frac=0.12):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * overlap_frac))
    cs = sp.split_documents(pages)
    s = Chroma.from_documents(cs, embedder, collection_name=f"c{size}", collection_metadata={"hnsw:space": "cosine"})
    out = {"chunk_size": size, "n_chunks": len(cs)}
    for k in ks:
        found = [any(norm(it["keyword"]) in norm(d.page_content) for d in s.similarity_search(it["question"], k=k)) for it in qa]
        out[f"hit@{k}"] = round(sum(found) / len(qa), 2)
    return out

pd.DataFrame([hit_rates(size) for size in [200, 400, 800, 1600]])

,chunk_size,n_chunks,hit@1,hit@4
0,200,912,0.17,0.33
1,400,446,0.33,0.50
2,800,230,0.08,0.58
3,1600,124,0.08,0.67


**What to notice:** hit@4 is higher than hit@1 for every chunk size, as it must be, and it grows with chunk size (0.33 at 200 characters up to 0.67 at 1600) because a bigger chunk simply contains more words, so the keyword is more likely to be inside. For hit@1 the picture is the opposite: the smallest chunks are best focused and bigger ones are diluted, so the ranking is noisy. Bigger is not free either: larger chunks send more irrelevant text to the LLM, which RAGAs will measure in notebook 03 as context precision. Overall, even the best setting finds the keyword for only about two thirds of the questions with this small embedding model, which is the room for improvement that RAG engineers spend their time on.

With only 12 questions, a difference of one question (0.08) is within noise. Treat these numbers as direction, not proof.

## 9. Save the final knowledge base

Notebook 02 loads the database from disk instead of rebuilding it. We already created `chroma_db/` in section 4 (chunk size 800, overlap 100, cosine). Let us confirm that a fresh Chroma client can reopen it and answer a query.

In [25]:
del store
reopened = Chroma(collection_name="rag_papers", embedding_function=embedder, persist_directory=DB_DIR)
print("chunks on disk:", reopened._collection.count())
print(reopened.similarity_search("What is the Dense Passage Retriever?", k=1)[0].page_content[:200].replace("\n", " "))

chunks on disk: 237
total ngrams generated by different models. Table 5 shows that RAG-Sequence’s generations are more diverse than RAG-Token’s, and both are significantly more diverse than BART without needing any diver


## Summary

| Stage | Library object | Key knobs | What we inspected |
|---|---|---|---|
| Load | `PyPDFLoader` | - | pages, metadata, ligature problem |
| Split | `RecursiveCharacterTextSplitter` | `chunk_size`, `chunk_overlap` | chunk length table, overlap text |
| Embed | `HuggingFaceEmbeddings` | model, `max_seq_length` | 384-dim unit vectors, similarity matrix |
| Store | `Chroma` | `hnsw:space`, `persist_directory` | raw ids, documents, metadata, vectors |
| Search | `similarity_search_with_score`, MMR, filters | `k`, `fetch_k`, `lambda_mult`, `filter` | distances, ranking across metrics |

**Next:** Notebook 02 connects this retriever to Gemini and builds the generation half of RAG.